In [1]:
# --- paste into your notebook and run ---
import os, json, math
from PIL import Image
import numpy as np
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools import mask as mask_util
from skimage.measure import regionprops, label, perimeter, regionprops_table
import pandas as pd
from tqdm import tqdm

# ---------- USER PATHS: replace these or set them before running ----------
MASKRCNN_WEIGHTS_PATH = r"C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\analysis\feature_json_trained_maskrcnn.pth"   # 
OUT_CSV_PATH = r"GB_training_data.csv"     
JSON_FILE_PATH = r"C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\json_2508\patches_coco_with_features.json"
IMAGE_FOLDER = r"C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\processed data\patches_2508"


In [2]:
# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MASK_THRESH = 0.5   # binarize predicted masks at 0.5
IOU_THRESH = 0.3     # as requested
NUM_OBJECT_CLASSES = 3   # other, rbc, cbc  (these are the object classes in your JSON)

# Note: the Mask-RCNN NUM_CLASSES used for training is NUM_CLASSES = 4 (including background)
MODEL_NUM_CLASSES = NUM_OBJECT_CLASSES + 1  # 4 (background + 3 object classes)

# ---------- utility functions ----------
def mask_iou_np(mask1, mask2):
    """IoU between two boolean masks (numpy arrays)."""
    inter = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def compute_region_features(mask_bool, image_gray):
    """
    mask_bool: HxW boolean numpy array
    image_gray: HxW numeric array (float or uint8). mean/std computed inside mask
    Returns dict with requested features
    """
    # ensure uint8 label image for regionprops
    lab = mask_bool.astype(np.uint8)
    # compute basic regionprops via skimage
    rp = regionprops(lab, intensity_image=image_gray)
    if len(rp) == 0:
        return None
    r = rp[0]
    area = float(r.area)
    minr, minc, maxr, maxc = r.bbox
    w = float(maxc - minc)
    h = float(maxr - minr)
    aspect_ratio = (w / h) if h > 0 else 0.0
    # perimeter: try skimage's perimeter if available; fallback to regionprops perimeter if present
    try:
        per = float(r.perimeter)
    except Exception:
        per = float(perimeter(mask_bool, neighbourhood=8))
    circularity = (4 * math.pi * area / (per * per)) if per > 0 else 0.0
    solidity = float(r.solidity) if hasattr(r, 'solidity') else 0.0
    # intensity stats inside mask
    pixels = image_gray[mask_bool]
    mean_intensity = float(np.mean(pixels)) if pixels.size > 0 else 0.0
    std_intensity = float(np.std(pixels)) if pixels.size > 0 else 0.0

    return {
        "original_area": area,
        "aspect_ratio": aspect_ratio,
        "circularity": circularity,
        "solidity": solidity,
        "mean_intensity": mean_intensity,
        "std_intensity": std_intensity,
        "bbox_w": w,
        "bbox_h": h
    }


In [3]:

# ---------- load Mask R-CNN (same arch you trained) ----------
print("Loading Mask R-CNN model...")
# instantiate base model (no external weights)
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, pretrained=False)
# replace heads
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, MODEL_NUM_CLASSES)
in_channels_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_channels_mask, hidden_layer, MODEL_NUM_CLASSES)
# load weights
model.load_state_dict(torch.load(MASKRCNN_WEIGHTS_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Model loaded and ready on", DEVICE)


Loading Mask R-CNN model...


C:\Users\shiva\AppData\Local\Temp\ipykernel_8104\783075874.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MASKRCNN_WEIGHTS_PATH, map_l

Model loaded and ready on cuda


In [4]:

# ---------- load COCO JSON ----------
with open(JSON_FILE_PATH, 'r') as f:
    coco = json.load(f)

# prepare lookup of GT annotations per image_id
anns_by_image = {}
for ann in coco['annotations']:
    img_id = ann['image_id']
    anns_by_image.setdefault(img_id, []).append(ann)

# some metadata lists
rows = []
n_images = len(coco.get('images', []))
print(f"Processing {n_images} images...")


Processing 772 images...


In [5]:
# we will iterate through images
for img_meta in tqdm(coco['images']):
    img_id = img_meta['id']
    # robust filename handling
    img_name = os.path.basename(img_meta['file_name'])
    img_path = os.path.join(IMAGE_FOLDER, img_name)
    if not os.path.exists(img_path):
        print(f"WARNING: image not found: {img_path}. Skipping.")
        continue

    # load image
    pil = Image.open(img_path).convert("RGB")
    img_arr = np.array(pil)   # H, W, 3
    # convert to grayscale for intensity-based features
    if img_arr.ndim == 3:
        # simple luminance transform
        img_gray = (0.2989 * img_arr[:,:,0] + 0.5870 * img_arr[:,:,1] + 0.1140 * img_arr[:,:,2]).astype(np.float32)
    else:
        img_gray = img_arr.astype(np.float32)

    # prepare GT masks (decoded via pycocotools) list
    gt_anns = anns_by_image.get(img_id, [])
    gt_masks = []
    gt_labels = []
    for ann in gt_anns:
        seg = ann.get('segmentation')
        if isinstance(seg, dict) and 'counts' in seg:
            # RLE dict
            gt_mask = mask_util.decode(seg)  # returns HxW (uint8)
        else:
            # Could be polygon or other; try mask_util.frPyObjects
            try:
                rles = mask_util.frPyObjects(seg, img_meta['height'], img_meta['width'])
                rle = mask_util.merge(rles)
                gt_mask = mask_util.decode(rle)
            except Exception:
                # skip if cannot decode
                continue
        # ensure boolean mask
        gt_masks.append((gt_mask.astype(bool)))
        gt_labels.append(int(ann['category_id']))   # keep ground-truth category_id as label (0/1/2)

    # run maskrcnn inference
    transform = torchvision.transforms.functional.to_tensor(pil).to(DEVICE)
    with torch.no_grad():
        outputs = model([transform])
    out = outputs[0]
    pred_masks = out.get('masks')  # tensor (N,1,H,W)
    pred_labels = out.get('labels').cpu().numpy() if out.get('labels') is not None else np.array([])
    pred_scores = out.get('scores').cpu().numpy() if out.get('scores') is not None else np.array([])

    if pred_masks is None or len(pred_masks) == 0:
        continue

    pred_masks_np = pred_masks.cpu().numpy()  # (N,1,H,W)

    # iterate predicted instances
    for p_idx in range(pred_masks_np.shape[0]):
        mask_prob = pred_masks_np[p_idx, 0]  # H x W floats
        mask_bin = mask_prob >= MASK_THRESH
        if mask_bin.sum() == 0:
            continue

        # compute IoU vs each GT mask and find best match
        best_iou = 0.0
        best_gt_idx = None
        for g_idx, gmask in enumerate(gt_masks):
            # ensure same shape; if shapes mismatch, try to resize or skip (we assume same)
            if gmask.shape != mask_bin.shape:
                # attempt to resize GT mask to predicted mask size only if shapes differ (rare)
                # skip this pred if shapes mismatch to avoid wrong IoU
                continue
            iou = mask_iou_np(mask_bin, gmask)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g_idx

        # drop false positives
        if best_iou < IOU_THRESH or best_gt_idx is None:
            continue

        # matched to GT: get GT label
        gt_label = gt_labels[best_gt_idx]   # keep original category_id (0/1/2)
        # compute features from regionprops on predicted mask using grayscale image
        feats = compute_region_features(mask_bin, img_gray)
        if feats is None:
            continue

        # Mask R-CNN predicted label and score
        pred_label_raw = int(pred_labels[p_idx])  # NOTE: torchvision uses labels 1..C (background excluded)
        pred_score = float(pred_scores[p_idx])

        # Map model's predicted label to the original category id space:
        # If during training you used target['labels'] = category_id + 1, then predicted label-1 -> original category_id
        pred_label_mapped = pred_label_raw - 1

        # build mrcnn probability vector by placing `pred_score` at predicted class and 
        # distributing remainder uniformly among other classes (simple, stable approach)
        probs = np.full((NUM_OBJECT_CLASSES,), (1.0 - pred_score) / (NUM_OBJECT_CLASSES - 1), dtype=float)
        # guard against numerical issues if NUM_OBJECT_CLASSES==1
        if NUM_OBJECT_CLASSES > 1 and 0 <= pred_label_mapped < NUM_OBJECT_CLASSES:
            probs[pred_label_mapped] = pred_score
        else:
            # unexpected label mapping: distribute uniform
            probs = np.full((NUM_OBJECT_CLASSES,), 1.0 / NUM_OBJECT_CLASSES)

        row = {
            "image_id": img_id,
            "image_name": img_name,
            "pred_index": p_idx,
            "pred_label_maskrcnn": pred_label_mapped,
            "pred_score_maskrcnn": pred_score,
            "iou_with_matched_gt": best_iou,
            "gt_label": gt_label,
            # requested regionprops features
            "original_area": feats["original_area"],
            "aspect_ratio": feats["aspect_ratio"],
            "circularity": feats["circularity"],
            "solidity": feats["solidity"],
            "mean_intensity": feats["mean_intensity"],
            "std_intensity": feats["std_intensity"],
            # mrcnn probs (3 classes: other, rbc, cbc) in that order
            "mrcnn_prob_other": float(probs[0]),
            "mrcnn_prob_rbc": float(probs[1]),
            "mrcnn_prob_cbc": float(probs[2]),
        }
        rows.append(row)

100%|██████████| 772/772 [3:05:04<00:00, 14.38s/it]     


In [6]:
# ---------- finalize DataFrame and save ----------
df = pd.DataFrame(rows)
print("Total rows (matched predicted instances kept):", len(df))
# save to CSV
out_dir = os.path.dirname(OUT_CSV_PATH)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)
df.to_csv(OUT_CSV_PATH, index=False)
print("Saved GB training CSV to:", OUT_CSV_PATH)


Total rows (matched predicted instances kept): 26366
Saved GB training CSV to: GB_training_data.csv
